# Huấn luyện mô hình phát hiện mụn (YOLO) cho sandwich beauty

Notebook này huấn luyện một mô hình **YOLO11n** phát hiện và khoanh vùng từng nốt mụn trên ảnh khuôn mặt, dựa trên bộ dữ liệu **ACNE04** (bản đã gán khung trên Roboflow Universe), rồi xuất ra **ONNX** để backend Node.js chạy trực tiếp.

**Trước khi chạy:**
1. Menu **Runtime → Change runtime type → T4 GPU**.
2. Tạo tài khoản Roboflow miễn phí, vào *Settings → API Keys* để lấy **Private API Key**.
3. Chạy lần lượt từng ô (Shift + Enter). Nên bật lưu vào Google Drive (ô cấu hình) để không mất kết quả nếu Colab ngắt kết nối.

**Kết quả cuối:** file `acne_yolo_export.zip` gồm `acne_yolo.onnx`, `acne_yolo.json` (tên lớp + chỉ số đánh giá) và các biểu đồ (`results.png`, `confusion_matrix.png`, `PR_curve.png`...) để đưa vào báo cáo.

In [ ]:
!nvidia-smi -L
!pip -q install "ultralytics>=8.3.0" roboflow onnx onnxslim

## 1. Cấu hình

In [ ]:
from getpass import getpass
import os

ROBOFLOW_API_KEY = getpass("Dán Roboflow Private API Key rồi Enter: ").strip()

# Bộ dữ liệu ACNE04 đã gán khung (4 lớp: nodules and cysts, papules, pustules, whitehead and blackhead)
# Trang: https://universe.roboflow.com/acne-vulgaris-detection/acne04-detection
WORKSPACE = "acne-vulgaris-detection"
PROJECT = "acne04-detection"
# Phương án khác (7 lớp): WORKSPACE = "andrei-dore-5lz05"; PROJECT = "acne04"

BASE_MODEL = "yolo11n.pt"   # bản nhỏ nhất, chạy nhanh trên CPU của server
IMGSZ = 640                  # nếu GPU đủ mạnh có thể thử 960 để bắt nốt mụn nhỏ tốt hơn
EPOCHS = 80
PATIENCE = 20                # dừng sớm nếu 20 epoch liền không cải thiện
BATCH = 16

SAVE_TO_DRIVE = True         # lưu kết quả vào Google Drive để không mất khi Colab ngắt
OUT_DIR = "/content/acne_runs"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/sandwich_beauty_acne_yolo"
os.makedirs(OUT_DIR, exist_ok=True)
print("Kết quả sẽ lưu tại:", OUT_DIR)

## 2. Tải bộ dữ liệu từ Roboflow (định dạng YOLO)

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)

dataset = None
for v in range(15, 0, -1):          # thử từ phiên bản mới nhất trở xuống
    try:
        dataset = project.version(v).download("yolov8", location=f"/content/datasets/{PROJECT}")
        DATASET_VERSION = v
        break
    except Exception as e:
        continue
assert dataset is not None, "Không tải được dataset — kiểm tra API key / tên project"
DATA_YAML = f"{dataset.location}/data.yaml"
print("Đã tải phiên bản", DATASET_VERSION, "→", DATA_YAML)
print(open(DATA_YAML).read())

In [ ]:
# Thống kê nhanh số ảnh và số nhãn theo từng tập — dùng cho phần mô tả dữ liệu trong báo cáo
import glob, yaml, collections
cfg = yaml.safe_load(open(DATA_YAML))
names = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"][k] for k in sorted(cfg["names"])]
stats = {}
for split in ["train", "valid", "test"]:
    labels = glob.glob(f"{dataset.location}/{split}/labels/*.txt")
    counter = collections.Counter()
    for path in labels:
        for line in open(path):
            if line.strip():
                counter[int(line.split()[0])] += 1
    stats[split] = {"images": len(labels), "boxes": sum(counter.values()), "per_class": {names[k]: n for k, n in sorted(counter.items())}}
for split, s in stats.items():
    print(f"{split:6s}: {s['images']:5d} ảnh, {s['boxes']:6d} nốt mụn  {s['per_class']}")

## 3. Huấn luyện

In [ ]:
from ultralytics import YOLO
model = YOLO(BASE_MODEL)
model.train(
    data=DATA_YAML, imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH,
    project=OUT_DIR, name="train", exist_ok=True,
    # Tăng cường dữ liệu nhẹ: ảnh khuôn mặt nên không lật dọc
    fliplr=0.5, flipud=0.0, degrees=5, hsv_v=0.3, mosaic=1.0, close_mosaic=10,
    seed=42, plots=True,
)
SAVE_DIR = str(model.trainer.save_dir)
BEST = f"{SAVE_DIR}/weights/best.pt"
print("Trọng số tốt nhất:", BEST)

## 4. Đánh giá độ chính xác

Đánh giá trên tập **test** (ảnh mô hình chưa từng thấy). Các chỉ số:
- **Precision**: trong các khung mô hình khoanh, bao nhiêu % đúng là mụn.
- **Recall**: trong các nốt mụn thật, mô hình tìm ra được bao nhiêu %.
- **mAP@50**: độ chính xác trung bình khi khung dự đoán trùng ≥ 50% với khung thật.
- **mAP@50-95**: trung bình trên nhiều ngưỡng trùng khớp (khắt khe hơn).

In [ ]:
import json, datetime
best = YOLO(BEST)
split = "test" if stats.get("test", {}).get("images") else "val"
m = best.val(data=DATA_YAML, imgsz=IMGSZ, split=split, project=OUT_DIR, name=f"eval_{split}", exist_ok=True, plots=True)
class_names = [best.names[i] for i in sorted(best.names)]
per_class = []
for i, idx in enumerate(m.box.ap_class_index):
    p, r, ap50, ap = m.box.class_result(i)
    per_class.append({"name": best.names[int(idx)], "precision": round(float(p), 4), "recall": round(float(r), 4), "map50": round(float(ap50), 4), "map50_95": round(float(ap), 4)})
metrics = {
    "split": split,
    "precision": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4),
    "map50": round(float(m.box.map50), 4), "map50_95": round(float(m.box.map), 4),
    "perClass": per_class,
}
print(json.dumps(metrics, indent=2, ensure_ascii=False))

## 5. Xuất ONNX cho backend

In [ ]:
onnx_path = best.export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True, dynamic=False)
print("ONNX:", onnx_path)

meta = {
    "name": "acne_yolo",
    "architecture": BASE_MODEL.replace(".pt", ""),
    "task": "detect",
    "outputLayout": "yolov8",           # [1, 4 + số lớp, số anchor], chưa NMS
    "imgsz": IMGSZ,
    "names": class_names,
    "dataset": {"source": f"roboflow:{WORKSPACE}/{PROJECT}/{DATASET_VERSION}", "base": "ACNE04 (Wu et al., ICCV 2019)", "stats": stats},
    "training": {"epochs": EPOCHS, "patience": PATIENCE, "batch": BATCH, "epochsRun": int(best.ckpt.get("epoch", -1)) + 1 if getattr(best, "ckpt", None) else None},
    "metrics": metrics,
    "trainedAt": datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z",
}
with open(f"{SAVE_DIR}/acne_yolo.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

## 6. Thử trên một ảnh test

In [ ]:
import random, matplotlib.pyplot as plt
test_imgs = glob.glob(f"{dataset.location}/{'test' if split == 'test' else 'valid'}/images/*")
sample = random.choice(test_imgs)
res = YOLO(onnx_path, task="detect")(sample, imgsz=IMGSZ, conf=0.25)[0]
print(len(res.boxes), "nốt mụn được phát hiện")
plt.figure(figsize=(8, 8)); plt.imshow(res.plot()[:, :, ::-1]); plt.axis("off")

## 7. Đóng gói và tải về

In [ ]:
import shutil, zipfile
from google.colab import files
pack = "/content/acne_yolo_export.zip"
eval_dir = f"{OUT_DIR}/eval_{split}"
with zipfile.ZipFile(pack, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(onnx_path, "acne_yolo.onnx")
    z.write(f"{SAVE_DIR}/acne_yolo.json", "acne_yolo.json")
    for name in ["results.png", "results.csv", "confusion_matrix.png", "confusion_matrix_normalized.png", "PR_curve.png", "F1_curve.png", "labels.jpg", "val_batch0_pred.jpg", "val_batch0_labels.jpg"]:
        for folder in (SAVE_DIR, eval_dir):
            path = f"{folder}/{name}"
            if os.path.exists(path):
                z.write(path, f"report/{os.path.basename(folder)}_{name}")
shutil.copy(pack, OUT_DIR)
files.download(pack)

## 8. Gắn vào website

1. Giải nén `acne_yolo_export.zip`, chép `acne_yolo.onnx` và `acne_yolo.json` vào `D:\SkinAdvisorShop\backend\ml\models\`.
2. Trong thư mục `backend`: `npm install` (đã khai báo sẵn `onnxruntime-node` và `sharp`).
3. Kiểm tra: `npm run acne:check -- duong_dan_anh.jpg`.
4. Khởi động lại backend. Trang Phân tích da sẽ tự dùng mô hình; nếu thiếu file model, hệ thống quay về chỉ dùng Gemini.

Thư mục `report/` trong zip chứa biểu đồ cho mục *Đánh giá độ chính xác mô hình* của báo cáo.